In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/config.json
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/pytorch_model-00002-of-00002.bin
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/tokenizer.json
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/tokenizer_config.json
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/pytorch_model.bin.index.json
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/pytorch_model-00001-of-00002.bin
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/special_tokens_map.json
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/.gitattributes
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/tokenizer.model
/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1/generation_config.json


In [2]:
import os

input_path = "/kaggle/input"
print("Available datasets in /kaggle/input/:")
print(os.listdir(input_path))


Available datasets in /kaggle/input/:
['mistral']


In [3]:
import huggingface_hub
print(huggingface_hub.__version__)
from huggingface_hub.utils import OfflineModeIsEnabled

0.27.0


In [4]:
import subprocess
import sys

packages = [
    
    "transformers",
    "datasets",
    "accelerate",
    "peft",
    "trl",
    "bitsandbytes",
    "wandb"
]

def install_missing_packages(packages):
    for package in packages:
        try:
            __import__(package)
            print(f"{package} is already installed.")
        except ImportError:
            print(f"{package} not found. Installing...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", package])

# Run the check and install process
install_missing_packages(packages)

transformers is already installed.
datasets is already installed.
accelerate is already installed.
peft is already installed.
trl is already installed.
bitsandbytes is already installed.
wandb is already installed.


In [5]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import (
    LoraConfig,
    PeftModel,
    prepare_model_for_kbit_training,
    get_peft_model,
)
import os, torch, wandb
from datasets import load_dataset
from trl import SFTTrainer, setup_chat_format

In [6]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HugFace")
wb_token = user_secrets.get_secret("wandb")
hf_token = user_secrets.get_secret("HugFace")
wb_token = user_secrets.get_secret("wandb")
login(token = hf_token)

In [7]:
wb_token = user_secrets.get_secret("wandb")
wandb.login(key=wb_token)

run = wandb.init(
    project='Fine-tune Mistral 7B Instruct on Medical Dataset', 
    job_type="training", 
    anonymous="allow"
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: trisha-sengupta-ece26 (trisha-sengupta-ece26-Heritage Institute of Technology). Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [8]:
base_model = "/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1"
dataset1 = load_dataset("Amod/mental_health_counseling_conversations")

new_model = "chat-doctor"

In [9]:
torch_dtype = torch.float16
attn_implementation = "eager"

In [10]:
from datasets import  Dataset
df = dataset1['train'].to_pandas()

# Reduce the number of rows
df_reduced = df.sample(n=2000, random_state=42)

# Convert back to Hugging Face dataset
dataset_reduced =  Dataset.from_pandas(df_reduced)

In [11]:
import os

path = "/kaggle/input/mistral/pytorch/7b-instruct-v0.1-hf/1"
if os.path.exists(path):
    print("Path exists. Listing files...")
    print(os.listdir(path))
else:
    print("Path does NOT exist!")


Path exists. Listing files...
['config.json', 'pytorch_model-00002-of-00002.bin', 'tokenizer.json', 'tokenizer_config.json', 'pytorch_model.bin.index.json', 'pytorch_model-00001-of-00002.bin', 'special_tokens_map.json', '.gitattributes', 'tokenizer.model', 'generation_config.json']


In [12]:
!pip install -U bitsandbytes

In [13]:
# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=attn_implementation
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [14]:
tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.padding_side = 'right'
tokenizer.chat_template = None
model, tokenizer = setup_chat_format(model, tokenizer)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [15]:
from peft import prepare_model_for_kbit_training

peft_config = LoraConfig(
    r=64,                     # Increased rank
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['up_proj', 'down_proj', 'gate_proj', 'k_proj', 'q_proj', 'v_proj', 'o_proj'],
    inference_mode=False,
)


In [16]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto",
    quantization_config=quantization_config
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [17]:
dataset_reduced

Dataset({
    features: ['Context', 'Response', '__index_level_0__'],
    num_rows: 2000
})

In [18]:
def format_chat_template(row):
    row_json = [{"role": "user", "content": row["Context"]},
               {"role": "assistant", "content": row["Response"]}]
    row["text"] = tokenizer.apply_chat_template(row_json, tokenize=False)
    return row

In [19]:
dataset = dataset_reduced.map(
    format_chat_template,
    num_proc=4,
)



Map (num_proc=4):   0%|          | 0/2000 [00:00<?, ? examples/s]

In [20]:
training_arguments = TrainingArguments(
    output_dir=new_model,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    num_train_epochs=3,
    evaluation_strategy="steps",
    eval_steps=50,
    logging_steps=10,
    warmup_ratio=0.1,
    learning_rate=5e-5,
    fp16=True,
    bf16=False,
    group_by_length=True,
    report_to="wandb",
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,    # Required for early stopping
    metric_for_best_model="loss"    # Monitor loss for early stopping
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [21]:
dataset = dataset.remove_columns(["Context", "Response", "__index_level_0__"])

In [22]:
tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.pad_token = tokenizer.eos_token
# Preprocess the dataset
def preprocess_function(examples):
    inputs = examples["Context"]
    targets = examples["Response"]
    model_inputs = tokenizer(inputs, padding="max_length", truncation=True, max_length=512)
    labels = tokenizer(targets, padding="max_length", truncation=True, max_length=512)
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset_reduced.map(preprocess_function, batched=True)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [23]:
from sklearn.model_selection import train_test_split

# Manually split the dataset into train and test
train_dataset, test_dataset = train_test_split(tokenized_datasets, test_size=0.2)

In [24]:
from datasets import Dataset

# Convert the split dictionaries back to Dataset objects
train_dataset = Dataset.from_dict(train_dataset)
test_dataset = Dataset.from_dict(test_dataset)

In [25]:
from transformers.trainer_callback import EarlyStoppingCallback

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=peft_config,
    tokenizer=tokenizer,
    args=training_arguments,
)

early_stopping = EarlyStoppingCallback(early_stopping_patience=3)
trainer.add_callback(early_stopping)

trainer.train()


<ipython-input-25-b6ea02d9bdda>:3: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a processing_class with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `processing_class.padding_side = 'right'` to your code.
  warnings.warn(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArgu

Step,Training Loss,Validation Loss
50,2.141200,2.104491
100,1.801600,1.740136
150,1.392100,1.481522
200,1.097300,1.317135
250,0.964800,1.214877
300,0.849100,1.154055


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tok

TrainOutput(global_step=300, training_loss=1.4807500171661376, metrics={'train_runtime': 19410.639, 'train_samples_per_second': 0.247, 'train_steps_per_second': 0.015, 'total_flos': 1.073248506740736e+17, 'train_loss': 1.4807500171661376, 'epoch': 3.0})

In [30]:
wandb.finish()
model.config.use_cache = True

In [31]:
messages = [
    {
        "role": "user",
        "content": "I often feel anxious in social situations. What are some ways to manage anxiety without medication?"
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, 
                                       add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt', padding=True, 
                   truncation=True, max_length=512).to("cuda")


outputs = model.generate(**inputs, max_length=250, num_return_sequences=1, num_beams=5, early_stopping=False, repetition_penalty=2.2)
print(outputs)
text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(text)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


tensor([[    1,     1,   733, 16289, 28793,   315,  2608,  1601, 19695,   297,
          2809, 11846, 28723,  1824,   460,   741,  4342,   298,  8594, 12758,
          1671, 20859, 28804,   733, 28748, 16289, 28793,  1387,   460,  2856,
          4342,   298,  8594, 12758,  1671, 20859, 28747,    13,    13, 28740,
         28723,   334,  3159,  2468,  1739, 13625,   282, 16481,  8264,   325,
         10957, 28738,  1329,   334, 12656,   349,   264,  1212,   302, 12238,
           369,  7263,   368,  9051,   304,  2268,  7087,  1654, 11533, 28723,
           661,   541,   347,  1215,  5645,   297, 17032, 12758, 28723,    13,
            13, 28750, 28723,  5855,   897,   352,  6772,  4030, 28747,  1387,
           460,  2856, 27607,  9804,   369,   541,  1316,  7643, 12758, 28725,
          1259,   390,  3534, 14232, 20858, 28725, 20458, 14540, 27607, 28725,
           304,  2273, 19965, 23976, 28723,    13,    13, 28770, 28723,  1529,
         25451, 28747, 28187,  9095,   541,  1316,  

In [32]:
trainer.push_to_hub()

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/trisha2710/chat-doctor/commit/8cbdc457911f5f32c58f5161ac17b80e01efeab6', commit_message='End of training', commit_description='', oid='8cbdc457911f5f32c58f5161ac17b80e01efeab6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/trisha2710/chat-doctor', endpoint='https://huggingface.co', repo_type='model', repo_id='trisha2710/chat-doctor'), pr_revision=None, pr_num=None)

In [33]:

trainer.model.save_pretrained(new_model)

In [ ]:
messages = [
    {
        "role": "user",
        "content": "How to get over trauma caused by bullying and harassment."
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, 
                                       add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt', padding=True, 
                   truncation=True, max_length=512).to("cuda")


outputs = model.generate(**inputs, max_length=250, num_return_sequences=1, num_beams=5, early_stopping=False, repetition_penalty=2.2)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(text)